In [1]:
import json
import re
import os
import random
from pathlib import Path

import numpy as np
import tensorflow as tf
from tensorflow import keras
from tensorflow.keras import layers,regularizers
from sklearn.model_selection import train_test_split

I0000 00:00:1792162324.428721   10556 port.cc:153] oneDNN custom operations are on. You may see slightly different numerical results due to floating-point round-off errors from different computation orders. To turn them off, set the environment variable `TF_ENABLE_ONEDNN_OPTS=0`.
I0000 00:00:1792162324.733930   10556 cpu_feature_guard.cc:227] This TensorFlow binary is optimized to use available CPU instructions in performance-critical operations.
To enable the following instructions: AVX2 AVX_VNNI FMA, in other operations, rebuild TensorFlow with the appropriate compiler flags.
I0000 00:00:1792162326.683049   10556 port.cc:153] oneDNN custom operations are on. You may see slightly different numerical results due to floating-point round-off errors from different computation orders. To turn them off, set the environment variable `TF_ENABLE_ONEDNN_OPTS=0`.


In [2]:
SEED = 42

os.environ["PYTHONHASHSEED"] = str(SEED)
random.seed(SEED)
np.random.seed(SEED)
tf.random.set_seed(SEED)

print("TensorFlow:", tf.__version__)

TensorFlow: 2.21.0


In [3]:
PROJECT_PATH = Path(
    "/home/sudhanshu_bajpai/projects/Voice_assistant"
)

DATA_PATH = PROJECT_PATH / "data" / "intents_improved.json"
SRC_PATH = PROJECT_PATH / "src"

MODEL_PATH = PROJECT_PATH / "data" /"intent_model_bigru.keras"
METADATA_PATH = PROJECT_PATH / "data" / "intent_metadata_lstm.json"

print("Dataset:", DATA_PATH)
print("Model:", MODEL_PATH)
print("Dataset exists:", DATA_PATH.exists())

Dataset: /home/sudhanshu_bajpai/projects/Voice_assistant/data/intents_improved.json
Model: /home/sudhanshu_bajpai/projects/Voice_assistant/data/intent_model_bigru.keras
Dataset exists: True


In [4]:
with open(DATA_PATH, "r", encoding="utf-8") as f:
    data = json.load(f)

print("Number of intents:", len(data))
print()

for intent, examples in data.items():
    print(f"{intent}: {len(examples)} examples")

Number of intents: 61

OPEN_EDGE: 40 examples
OPEN_VSCODE: 40 examples
OPEN_NOTEPAD: 40 examples
OPEN_FILE_EXPLORER: 40 examples
OPEN_TERMINAL: 40 examples
OPEN_CALCULATOR: 40 examples
OPEN_SETTINGS: 40 examples
OPEN_TASK_MANAGER: 40 examples
OPEN_SNIPPING_TOOL: 40 examples
OPEN_PROJECT: 40 examples
START_JUPYTER: 40 examples
OPEN_PYTHON: 40 examples
OPEN_GIT_BASH: 40 examples
OPEN_POWERSHELL: 40 examples
OPEN_COMMAND_PROMPT: 40 examples
OPEN_VSCODE_TERMINAL: 40 examples
RUN_PYTHON_SCRIPT: 40 examples
SEARCH_WEB: 40 examples
OPEN_WEBSITE: 40 examples
OPEN_YOUTUBE: 40 examples
OPEN_GITHUB: 40 examples
OPEN_GMAIL: 40 examples
OPEN_GOOGLE_DRIVE: 40 examples
OPEN_CHATGPT: 40 examples
CHECK_CPU: 40 examples
CHECK_GPU: 40 examples
CHECK_RAM: 40 examples
CHECK_DISK: 40 examples
CHECK_WINDOWS_VERSION: 40 examples
CHECK_IP_ADDRESS: 40 examples
CHECK_BATTERY: 40 examples
CHECK_NETWORK: 40 examples
CHECK_PROCESS_LIST: 40 examples
LOCK_COMPUTER: 40 examples
SLEEP_COMPUTER: 40 examples
MUTE_AUDIO: 

In [5]:
def tokenize(text):
    return re.findall(
        r"[a-zA-Z0-9']+",
        text.lower()
    )

In [6]:
print(tokenize("Open my programming environment"))
print(tokenize("Search for TensorFlow GPU setup"))

['open', 'my', 'programming', 'environment']
['search', 'for', 'tensorflow', 'gpu', 'setup']


In [7]:
PAD_TOKEN = "<PAD>"
UNK_TOKEN = "<UNK>"

vocab = {
    PAD_TOKEN: 0,
    UNK_TOKEN: 1
}

all_words = set()

for examples in data.values():

    for sentence in examples:

        tokens = tokenize(sentence)

        for token in tokens:
            all_words.add(token)

for word in sorted(all_words):
    vocab[word] = len(vocab)

print("Vocabulary size:", len(vocab))

print()
print("First 20 vocabulary entries:")

for word, index in list(vocab.items())[:20]:
    print(word, "→", index)

Vocabulary size: 935

First 20 vocabulary entries:
<PAD> → 0
<UNK> → 1
10 → 2
11 → 3
12 → 4
2026 → 5
25 → 6
3 → 7
30 → 8
45 → 9
5050 → 10
a → 11
about → 12
access → 13
accuracy → 14
activate → 15
active → 16
adapter → 17
adapters → 18
add → 19


In [8]:
intents = list(data.keys())

intent_to_id = {
    intent: i
    for i, intent in enumerate(intents)
}

id_to_intent = {
    i: intent
    for intent, i in intent_to_id.items()
}

print("Intent mapping:")

for intent, index in intent_to_id.items():
    print(index, "→", intent)

Intent mapping:
0 → OPEN_EDGE
1 → OPEN_VSCODE
2 → OPEN_NOTEPAD
3 → OPEN_FILE_EXPLORER
4 → OPEN_TERMINAL
5 → OPEN_CALCULATOR
6 → OPEN_SETTINGS
7 → OPEN_TASK_MANAGER
8 → OPEN_SNIPPING_TOOL
9 → OPEN_PROJECT
10 → START_JUPYTER
11 → OPEN_PYTHON
12 → OPEN_GIT_BASH
13 → OPEN_POWERSHELL
14 → OPEN_COMMAND_PROMPT
15 → OPEN_VSCODE_TERMINAL
16 → RUN_PYTHON_SCRIPT
17 → SEARCH_WEB
18 → OPEN_WEBSITE
19 → OPEN_YOUTUBE
20 → OPEN_GITHUB
21 → OPEN_GMAIL
22 → OPEN_GOOGLE_DRIVE
23 → OPEN_CHATGPT
24 → CHECK_CPU
25 → CHECK_GPU
26 → CHECK_RAM
27 → CHECK_DISK
28 → CHECK_WINDOWS_VERSION
29 → CHECK_IP_ADDRESS
30 → CHECK_BATTERY
31 → CHECK_NETWORK
32 → CHECK_PROCESS_LIST
33 → LOCK_COMPUTER
34 → SLEEP_COMPUTER
35 → MUTE_AUDIO
36 → UNMUTE_AUDIO
37 → INCREASE_VOLUME
38 → DECREASE_VOLUME
39 → SCREENSHOT
40 → OPEN_NOTIFICATION_SETTINGS
41 → TOGGLE_WIFI
42 → OPEN_DOWNLOADS
43 → OPEN_DOCUMENTS
44 → OPEN_DESKTOP
45 → OPEN_PICTURES
46 → CREATE_FOLDER
47 → CREATE_TEXT_FILE
48 → MAKE_NOTES
49 → OPEN_NOTES
50 → SET_TIMER
51 

In [9]:
MAX_LEN =900 
def encode_sentence(sentence):

    tokens = tokenize(sentence)

    ids = []

    for token in tokens:
        ids.append(
            vocab.get(
                token,
                vocab[UNK_TOKEN]
            )
        )

    # Truncate
    ids = ids[:MAX_LEN]

    # Pad
    while len(ids) < MAX_LEN:
        ids.append(vocab[PAD_TOKEN])

    return ids

example = "open my programming environment"

encoded = encode_sentence(example)

print("Sentence:", example)
print("Encoded:", encoded)
print("Length:", len(encoded))

Sentence: open my programming environment
Encoded: [540, 500, 605, 243, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0

In [10]:
samples = []

for intent, examples in data.items():

    label = intent_to_id[intent]

    for sentence in examples:

        encoded = encode_sentence(sentence)

        samples.append(
            (encoded, label)
        )

print("Total samples:", len(samples))

Total samples: 2520


In [11]:
SEED = 42

# Independent random generator for reproducible dataset splitting
rng = random.Random(SEED)

train_samples = []
val_samples = []

for intent, examples in data.items():

    label = intent_to_id[intent]

    examples_copy = examples.copy()

    # Reproducible shuffle
    rng.shuffle(examples_copy)

    split_index = int(0.8 * len(examples_copy))

    train_examples = examples_copy[:split_index]
    val_examples = examples_copy[split_index:]

    for sentence in train_examples:
        train_samples.append(
            (
                encode_sentence(sentence),
                label
            )
        )

    for sentence in val_examples:
        val_samples.append(
            (
                encode_sentence(sentence),
                label
            )
        )

# Reproducible final ordering
rng.shuffle(train_samples)
rng.shuffle(val_samples)

print("Training samples:", len(train_samples))
print("Validation samples:", len(val_samples))

X_train = np.array(
    [x for x, y in train_samples],
    dtype=np.int32
)

y_train = np.array(
    [y for x, y in train_samples],
    dtype=np.int32
)

X_val = np.array(
    [x for x, y in val_samples],
    dtype=np.int32
)

y_val = np.array(
    [y for x, y in val_samples],
    dtype=np.int32
)

print("X_train:", X_train.shape)
print("y_train:", y_train.shape)

print("X_val:", X_val.shape)
print("y_val:", y_val.shape)

Training samples: 2016
Validation samples: 504
X_train: (2016, 900)
y_train: (2016,)
X_val: (504, 900)
y_val: (504,)


In [12]:
BATCH_SIZE = 16

train_dataset = (
    tf.data.Dataset
    .from_tensor_slices(
        (X_train, y_train)
    )
    .shuffle(
        len(X_train),
        seed=SEED
    )
    .batch(BATCH_SIZE)
)

val_dataset = (
    tf.data.Dataset
    .from_tensor_slices(
        (X_val, y_val)
    )
    .batch(BATCH_SIZE)
)

W0000 00:00:1792162332.290823   10556 gpu_device.cc:2459] TensorFlow was not built with CUDA kernel binaries compatible with compute capability 12.0a. CUDA kernels will be jit-compiled from PTX, which could take 30 minutes or longer.
W0000 00:00:1792162332.295107   10556 gpu_device.cc:2459] TensorFlow was not built with CUDA kernel binaries compatible with compute capability 12.0a. CUDA kernels will be jit-compiled from PTX, which could take 30 minutes or longer.
I0000 00:00:1792162332.499114   10556 gpu_device.cc:2043] Created device /job:localhost/replica:0/task:0/device:GPU:0 with 5262 MB memory:  -> device: 0, name: NVIDIA GeForce RTX 5050 Laptop GPU, pci bus id: 0000:01:00.0, compute capability: 12.0a


In [24]:
model = keras.Sequential([
    layers.Input(shape=(MAX_LEN,)),

    layers.Embedding(
        input_dim=len(vocab),
        output_dim=64,
        mask_zero=True
    ),

    layers.Bidirectional(
        layers.LSTM(
            64,
            dropout=0.4,
            return_sequences=True,
            kernel_regularizer=regularizers.l2(1e-4)
        )
    ),

    layers.Bidirectional(
            layers.LSTM(
                32,
                dropout=0.4,
                kernel_regularizer=regularizers.l2(1e-4)
            )
        ),

    layers.Dense(
        32,
        activation="relu",
        kernel_regularizer=regularizers.l2(1e-4)
    ),

    layers.Dense(len(intents), activation="softmax")
])

model.compile(
    optimizer=keras.optimizers.Adam(learning_rate=0.001),
    loss="sparse_categorical_crossentropy",
    metrics=["accuracy"]
)

# model = keras.Sequential([
#     layers.Input(shape=(MAX_LEN,)),

#     layers.Embedding(
#         input_dim=len(vocab),
#         output_dim=64,
#         mask_zero=True
#     ),

#     layers.Bidirectional(
#         layers.GRU(
#             64,
#             kernel_regularizer=regularizers.l2(1e-4)
#         )
#     ),

#     layers.Dense(
#         64,
#         activation="relu",
#         kernel_regularizer=regularizers.l2(1e-4)
#     ),

#     layers.Dense(len(intents), activation="softmax")
# ])

# model.compile(
#     optimizer=keras.optimizers.Adam(learning_rate=0.001),
#     loss="sparse_categorical_crossentropy",
#     metrics=["accuracy"]
# )

model.summary()

Model: "sequential_1"

┏━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━┳━━━━━━━━━━━━━━━━━━━━━━━━┳━━━━━━━━━━━━━━━┓
┃ Layer (type)                    ┃ Output Shape           ┃       Param # ┃
┡━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━╇━━━━━━━━━━━━━━━━━━━━━━━━╇━━━━━━━━━━━━━━━┩
│ embedding_1 (Embedding)         │ (None, 900, 64)        │        59,840 │
├─────────────────────────────────┼────────────────────────┼───────────────┤
│ bidirectional_2 (Bidirectional) │ (None, 900, 128)       │        66,048 │
├─────────────────────────────────┼────────────────────────┼───────────────┤
│ bidirectional_3 (Bidirectional) │ (None, 64)             │        41,216 │
├─────────────────────────────────┼────────────────────────┼───────────────┤
│ dense_2 (Dense)                 │ (None, 32)             │         2,080 │
├─────────────────────────────────┼────────────────────────┼───────────────┤
│ dense_3 (Dense)                 │ (None, 61)             │         2,013 │
└─────────────────────────────────┴────────────────────────┴───────────────┘

 Total params: 171,197 (668.74 KB)

 Trainable params: 171,197 (668.74 KB)

 Non-trainable params: 0 (0.00 B)

In [25]:
EPOCHS = 100

early_stopping = keras.callbacks.EarlyStopping(
    monitor="val_loss",
    patience=5,
    restore_best_weights=True
)

history = model.fit(
    train_dataset,
    validation_data=val_dataset,
    epochs=EPOCHS
)
# early_stopping = keras.callbacks.EarlyStopping(
#     monitor="val_loss",
#     patience=5,
#     restore_best_weights=True
# )

# history = model.fit(
#     X_train,
#     y_train,
#     validation_data=(X_val, y_val),
#     epochs=100,
#     batch_size=16,
#     callbacks=[early_stopping],
#     verbose=1
# )

Epoch 1/100
126/126 ━━━━━━━━━━━━━━━━━━━━ 5s 24ms/step - accuracy: 0.0506 - loss: 4.0896 - val_accuracy: 0.1210 - val_loss: 3.8595
Epoch 2/100
126/126 ━━━━━━━━━━━━━━━━━━━━ 3s 21ms/step - accuracy: 0.1642 - loss: 3.2702 - val_accuracy: 0.2956 - val_loss: 2.6658
Epoch 3/100
126/126 ━━━━━━━━━━━━━━━━━━━━ 3s 22ms/step - accuracy: 0.3681 - loss: 2.2114 - val_accuracy: 0.5119 - val_loss: 1.8732
Epoch 4/100
126/126 ━━━━━━━━━━━━━━━━━━━━ 3s 23ms/step - accuracy: 0.5476 - loss: 1.5984 - val_accuracy: 0.6627 - val_loss: 1.3786
Epoch 5/100
126/126 ━━━━━━━━━━━━━━━━━━━━ 3s 24ms/step - accuracy: 0.6944 - loss: 1.1424 - val_accuracy: 0.7659 - val_loss: 1.0744
Epoch 6/100
126/126 ━━━━━━━━━━━━━━━━━━━━ 3s 23ms/step - accuracy: 0.7917 - loss: 0.8595 - val_accuracy: 0.7976 - val_loss: 0.9184
Epoch 7/100
126/126 ━━━━━━━━━━━━━━━━━━━━ 3s 22ms/step - accuracy: 0.8472 - loss: 0.6660 - val_accuracy: 0.8492 - val_loss: 0.7892
Epoch 8/100
126/126 ━━━━━━━━━━━━━━━━━━━━ 3s 22ms/step - accuracy: 0.8869 - loss: 0.5375 - 

In [26]:
val_loss, val_accuracy = model.evaluate(
    val_dataset,
    verbose=0
)

print("Validation loss:", val_loss)
print("Validation accuracy:", val_accuracy)

Validation loss: 0.6127222180366516
Validation accuracy: 0.9265872836112976


In [27]:
model.save(MODEL_PATH)

print("LSTM model saved to:")
print(MODEL_PATH)


LSTM model saved to:
/home/sudhanshu_bajpai/projects/Voice_assistant/data/intent_model_bigru.keras


In [28]:
metadata = {
    "vocab": vocab,
    "intent_to_id": intent_to_id,
    "id_to_intent": id_to_intent,
    "max_len": MAX_LEN
}

with open(METADATA_PATH, "w") as f:
    json.dump(metadata, f, indent=4)

print("LSTM metadata saved to:")
print(METADATA_PATH)

LSTM metadata saved to:
/home/sudhanshu_bajpai/projects/Voice_assistant/data/intent_metadata_lstm.json


In [29]:
hard_test_commands = {
    "OPEN_VSCODE": [
        "I want to start coding",
        "bring up my coding workspace",
        "I need somewhere to write code",
        "get my development workspace ready",
        "open the editor I use for programming",
    ],

    "CHECK_CPU": [
        "how hard is my processor working",
        "show me the processor load",
        "what is my CPU doing",
        "tell me how much processor I'm using",
    ],

    "CHECK_GPU": [
        "how busy is my graphics card",
        "show the graphics processor usage",
        "tell me what my GPU is doing",
        "how much graphics processing is being used",
    ],

    "SEARCH_WEB": [
        "find this online",
        "look this up for me",
        "can you find that on the internet",
        "I need information about something",
    ],

    "MAKE_NOTES": [
        "put this in my notes",
        "save this for me",
        "write this down for later",
        "remember this in a note",
    ],

    "STOP_ASSISTANT": [
        "you can stop listening now",
        "end voice mode",
        "quit listening",
        "terminate the session",
    ]
}

In [30]:
def predict_intent(text):
    encoded = np.asarray(encode_sentence(text), dtype=np.int32)[None, :]
    probabilities = model.predict(encoded, verbose=0)[0]

    predicted_id = int(np.argmax(probabilities))
    return id_to_intent[predicted_id], float(probabilities[predicted_id])


for expected_intent, commands in hard_test_commands.items():
    print(f"\n===== {expected_intent} =====")

    for command in commands:
        intent, confidence = predict_intent(command)
        result = "✓" if intent == expected_intent else "✗"

        print(
            f"{result} "
            f"{command:45} -> "
            f"{intent:25} "
            f"{confidence * 100:.2f}%"
        )


===== OPEN_VSCODE =====
✓ I want to start coding                        -> OPEN_VSCODE               98.81%
✓ bring up my coding workspace                  -> OPEN_VSCODE               99.03%
✗ I need somewhere to write code                -> MAKE_NOTES                75.27%
✗ get my development workspace ready            -> OPEN_PROJECT              98.84%
✓ open the editor I use for programming         -> OPEN_VSCODE               99.97%

===== CHECK_CPU =====
✓ how hard is my processor working              -> CHECK_CPU                 99.98%
✓ show me the processor load                    -> CHECK_CPU                 99.96%
✓ what is my CPU doing                          -> CHECK_CPU                 100.00%
✓ tell me how much processor I'm using          -> CHECK_CPU                 99.79%

===== CHECK_GPU =====
✓ how busy is my graphics card                  -> CHECK_GPU                 99.97%
✓ show the graphics processor usage             -> CHECK_GPU                 99.84%
✓ te

In [31]:
print("Final training accuracy:",
      history.history["accuracy"][-1])

print("Final validation accuracy:",
      history.history["val_accuracy"][-1])

print("Final training loss:",
      history.history["loss"][-1])

print("Final validation loss:",
      history.history["val_loss"][-1])

Final training accuracy: 0.9970238208770752
Final validation accuracy: 0.9265872836112976
Final training loss: 0.058964040130376816
Final validation loss: 0.6127222180366516


In [32]:
best_epoch = np.argmin(history.history["val_loss"])

print("Best epoch:", best_epoch + 1)
print(
    "Training accuracy:",
    round(history.history["accuracy"][best_epoch], 4)
)
print(
    "Validation accuracy:",
    round(history.history["val_accuracy"][best_epoch], 4)
)
print(
    "Validation loss:",
    round(history.history["val_loss"][best_epoch], 4)
)

Best epoch: 78
Training accuracy: 0.994
Validation accuracy: 0.9365
Validation loss: 0.4818
